# BaseAttentive: Quick Start Guide

This notebook demonstrates how to create and configure a BaseAttentive model.

## What is BaseAttentive?

BaseAttentive is a foundational blueprint for building powerful, data-driven, sequence-to-sequence time series forecasting models with advanced attention mechanisms.

### Key Features
- **Flexible Architecture**: Hybrid (LSTM+Attention) or Pure Transformer
- **Multiple Attention Types**: Cross, Hierarchical, Memory-Augmented
- **Feature Selection**: Variable Selection Networks (VSN) for learnable feature importance
- **Uncertainty Quantification**: Quantile-based forecasting for confidence intervals
- **Dynamic Time Warping**: Automatic temporal alignment

## Step 1: Installation

```bash
pip install base-attentive==2.2.0
```

Or for development:
```bash
git clone https://github.com/earthai-tech/base-attentive.git
cd base-attentive
pip install -e ".[dev]"
```

> **v2.2.0 note:** `BASE_ATTENTIVE_BACKEND` must be set before importing
> `base_attentive`. Run the **Backend Setup** cell above first.

In [ ]:
# ── v2.2.0 Backend Setup ─────────────────────────────────────────────────────
# BASE_ATTENTIVE_BACKEND must be set *before* importing base_attentive.
# Choose your installed backend: "tensorflow" | "torch" | "jax" | "auto"
import os
os.environ.setdefault("BASE_ATTENTIVE_BACKEND", "tensorflow")
os.environ.setdefault("KERAS_BACKEND", os.environ["BASE_ATTENTIVE_BACKEND"])
BACKEND = os.environ["BASE_ATTENTIVE_BACKEND"]
print(f"Backend: {BACKEND}")

In [ ]:
# Import required libraries
import numpy as np

from base_attentive import BaseAttentive, __version__

print(f"BaseAttentive version: {__version__}")

## Step 2: Create a Basic Model

Define input dimensions and create a model instance.

In [ ]:
# Define input dimensions
STATIC_DIM = (
    4  # e.g., location, altitude, climate zone, soil type
)
DYNAMIC_DIM = 8  # e.g., temperature, humidity, pressure, wind speed, etc
FUTURE_DIM = 6  # e.g., forecasted weather features
OUTPUT_DIM = 2  # e.g., target 1 and target 2
FORECAST_HORIZON = 24  # Predict 24 steps ahead

# Create model
model = BaseAttentive(
    static_input_dim=STATIC_DIM,
    dynamic_input_dim=DYNAMIC_DIM,
    future_input_dim=FUTURE_DIM,
    output_dim=OUTPUT_DIM,
    forecast_horizon=FORECAST_HORIZON,
    embed_dim=32,
    attention_units=64,
    num_heads=8,
    dropout_rate=0.15,
    name="BasicAttentiveModel",
)

print("✅ Model created successfully!")
print(model)

## Step 3: Inspect Model Configuration

View the complete model configuration.

In [ ]:
# Get full configuration
config = model.get_config()

print("\n📋 Model Configuration:")
print("=" * 50)
for key, value in config.items():
    if value is not None:
        print(f"  {key:.<30} {value}")

## Step 4: Create Example Data

Generate synthetic data to demonstrate the model structure.

In [ ]:
# Generate sample data
BATCH_SIZE = 32
TIME_STEPS = 10  # Historical lookback window

# Static features: (batch_size, static_dim)
static_features = np.random.randn(
    BATCH_SIZE, STATIC_DIM
).astype("float32")

# Dynamic features: (batch_size, time_steps, dynamic_dim)
dynamic_features = np.random.randn(
    BATCH_SIZE, TIME_STEPS, DYNAMIC_DIM
).astype("float32")

# Future features: (batch_size, forecast_horizon, future_dim)
future_features = np.random.randn(
    BATCH_SIZE, FORECAST_HORIZON, FUTURE_DIM
).astype("float32")

print("\n📊 Sample Data Shapes:")
print(f"  Static:  {static_features.shape}")
print(f"  Dynamic: {dynamic_features.shape}")
print(f"  Future:  {future_features.shape}")

## Step 5: Model Summary

Display model hyperparameters and settings.

In [ ]:
# Display model summary
print("\n📈 Model Summary:")
print("=" * 50)

summary_info = {
    "Architecture": model.objective,
    "Mode": model.mode,
    "Encoder Layers": model.num_encoder_layers,
    "Attention Heads": model.num_heads,
    "Dropout Rate": model.dropout_rate,
    "Use Residuals": model.use_residuals,
    "Use VSN": model.use_vsn,
    "Apply DTW": model.apply_dtw,
}

for key, value in summary_info.items():
    print(f"  {key:.<25} {value}")

## Step 6: Save/Load Model Configuration

Demonstrate configuration serialization.

In [ ]:
# Save configuration
import json

config = model.get_config()

# Convert to JSON-serializable format
json_config = json.dumps(
    {
        k: str(v)
        if not isinstance(v, (int, float, bool, type(None)))
        else v
        for k, v in config.items()
    },
    indent=2,
)

print("\n💾 Saved Configuration (JSON snippet):")
print(json_config[:500] + "...")

# Later, recreate model from config
model2 = BaseAttentive.from_config(config)
print("\n✅ Model recreated from configuration")

## Step 5: Train the Model

Compile and fit the model on our synthetic data for a quick demonstration.

In [ ]:
import keras

# Use a small, quick-to-train setup
BATCH_SIZE = 32
TIME_STEPS = 10

# Regenerate data with consistent seed
np.random.seed(42)
static_data  = np.random.randn(BATCH_SIZE, STATIC_DIM).astype('float32')
dynamic_data = np.random.randn(BATCH_SIZE, TIME_STEPS, DYNAMIC_DIM).astype('float32')
future_data  = np.random.randn(BATCH_SIZE, FORECAST_HORIZON, FUTURE_DIM).astype('float32')

# Synthetic target: sine wave per sample + noise
t = np.linspace(0, 2 * np.pi, FORECAST_HORIZON)
target = (np.sin(t)[None, :, None] * np.random.rand(BATCH_SIZE, 1, OUTPUT_DIM)
          + 0.1 * np.random.randn(BATCH_SIZE, FORECAST_HORIZON, OUTPUT_DIM)).astype('float32')

model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])

history = model.fit(
    [static_data, dynamic_data, future_data],
    target,
    epochs=10,
    batch_size=16,
    validation_split=0.2,
    verbose=1,
)

print(f'
Final train MSE: {history.history["loss"][-1]:.4f}')
print(f'Final val   MSE: {history.history["val_loss"][-1]:.4f}')

## Step 6: Visualize Training History

Plot the training and validation loss curves.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history.history['loss'],     label='Train MSE', linewidth=2)
ax.plot(history.history['val_loss'], label='Val MSE',   linewidth=2, linestyle='--')
ax.set_title('Training History — BaseAttentive Quick Start', fontsize=13)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 7: Forecast vs Actual

Compare model predictions against the synthetic targets for a single sample.

In [ ]:
# Predict on the full batch
preds = model.predict([static_data, dynamic_data, future_data], verbose=0)
# preds shape: (batch, forecast_horizon, output_dim)

sample_idx = 0
fig, axes = plt.subplots(1, OUTPUT_DIM, figsize=(6 * OUTPUT_DIM, 4), squeeze=False)

hours = np.arange(1, FORECAST_HORIZON + 1)
for d in range(OUTPUT_DIM):
    ax = axes[0][d]
    ax.plot(hours, target[sample_idx, :, d],  label='Actual',    linewidth=2,   color='steelblue')
    ax.plot(hours, preds[sample_idx,  :, d],  label='Forecast',  linewidth=2,   color='tomato', linestyle='--')
    ax.fill_between(hours,
                    preds[sample_idx, :, d] - 0.1,
                    preds[sample_idx, :, d] + 0.1,
                    alpha=0.2, color='tomato', label='±0.1 band')
    ax.set_title(f'Output dim {d + 1}: Forecast vs Actual', fontsize=12)
    ax.set_xlabel('Forecast step (h)')
    ax.set_ylabel('Value')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('BaseAttentive — Quick Start Forecast', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

mae = float(np.mean(np.abs(preds[sample_idx] - target[sample_idx])))
print(f'Sample MAE: {mae:.4f}')

## What's Next?

Check out the other example notebooks:

1. **02_hybrid_vs_transformer.ipynb** - Compare hybrid and transformer encoder choices
2. **03_attention_stack_configuration.ipynb** - Configure the decoder attention stack
3. **04_standalone_applications.ipynb** - End-to-end forecasting examples across domains
4. **05_kernel_robust_networks.ipynb** - Use BaseAttentive as a reusable forecasting kernel
5. **06_crps_probabilistic_forecasting.ipynb** - Quantile, Gaussian, and mixture CRPS workflows
6. **07_v2_spec_registry.ipynb** - Declarative V2 specs, registries, and resolver assembly

## 📚 Resources

- [Full Documentation](https://github.com/earthai-tech/base-attentive)
- [GitHub Issues](https://github.com/earthai-tech/base-attentive/issues)
